# Day 4-2: Transfer Learning with ResNet50

**강의 시간**: 2시간  
**학습 목표**:
- ImageNet Pretrained ResNet50 활용
- Feature Extraction vs Fine-tuning
- Data Augmentation 적용
- 성능 향상 (Accuracy 90%+)

**사전 요구사항**: Day 4-1 완료  
**Day 4-1 성능**: Accuracy 85.42%, COVID Recall 95.71%

## 🔧 0. 환경 설정

In [ ]:
# 라이브러리 설치
%pip install -q 'mlflow>=2,<3' dagshub tensorflow opencv-python scikit-learn

print("✅ 라이브러리 설치 완료!")

In [ ]:
# 라이브러리 임포트
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

import mlflow
import dagshub

np.random.seed(42)
tf.random.set_seed(42)

print(f"✅ TensorFlow {tf.__version__}")
print(f"✅ GPU: {len(tf.config.list_physical_devices('GPU'))} devices")

In [ ]:
# 시각화 설정
sns.set_style('whitegrid')

# 1) 폰트 파일 직접 다운로드 (런타임 재시작 불필요)
!wget -q -O NanumGothic.ttf -L "https://fonts.gstatic.com/ea/nanumgothic/v5/NanumGothic-Regular.ttf"

import matplotlib.font_manager as fm

# 폰트 파일 경로
font_path = "NanumGothic.ttf"

# 폰트 매니저에 폰트 추가
fm.fontManager.addfont(font_path)

plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

# 폰트 속성 설정
font_prop = fm.FontProperties(fname=font_path)
plt.rcParams["font.family"] = font_prop.get_name()
plt.rcParams["axes.unicode_minus"] = False

🔥 이 부분은 수정이 필요합니다.

**repo_owner**와 **repo_name**을 본인의 Dagshub 정보로 채워 주세요.

In [ ]:
# MLflow 설정
import mlflow
import dagshub

repo_owner = # 🔥 직접 작성이 필요합니다.
repo_name  = # 🔥 직접 작성이 필요합니다.

dagshub.init(repo_owner=repo_owner, repo_name=repo_name, mlflow=True)
mlflow.set_experiment('day4-covid-xray-classification')
print('✅ MLflow 설정 완료!')


## 📂 1. 데이터 로드 (Day 4-1에서 계속)

## 📦 1. 데이터 다운로드

### Kaggle API 설정

**사전 준비**:
1. Kaggle 계정 생성 (https://www.kaggle.com)
2. Account → API → "Create New API Token"
3. `kaggle.json` 다운로드

In [ ]:
import os
from google.colab import userdata

# 1. [가장 중요] kaggle 라이브러리를 임포트하기 "전"에 환경 변수를 먼저 설정합니다.
# 보안 비밀에 KAGGLE_USERNAME(내 아이디)과 KAGGLE_API_TOKEN(키 문자열)이 있어야 합니다.
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_API_TOKEN')      # Python API용 표준 이름
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN') # 최신 토큰 방식 호환용

# 2. 환경 변수가 설정된 후 비로소 라이브러리를 불러옵니다.
try:
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate() # 이제 환경 변수를 인식하고 에러 없이 통과합니다.

    # 3. 데이터셋 다운로드 실행
    dataset_id = 'tawsifurrahman/covid19-radiography-database'
    print(f"📥 {dataset_id} 다운로드 시작...")

    api.dataset_download_files(
        dataset_id,
        path='./data',
        unzip=True,
        quiet=False
    )

    print("\n✅ 다운로드 및 압축 해제 완료!")

except Exception as e:
    print(f"\n❌ 오류 발생: {e}")

In [ ]:
# 데이터 구조 확인
data_dir = Path('./data/COVID-19_Radiography_Dataset')

In [ ]:
# 데이터 경로 수집 (메모리 효율적)
def get_file_paths_and_labels(data_dir):
    """이미지 경로와 라벨만 수집 (메모리 효율적)"""

    class_names = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}

    file_paths = []
    labels = []

    print("📋 파일 경로 수집 중...")

    for class_name in class_names:
        image_dir = data_dir / class_name / 'images'
        image_paths = list(image_dir.glob('*.png'))

        for img_path in image_paths:
            file_paths.append(str(img_path))
            labels.append(class_to_idx[class_name])

        print(f"  {class_name:20s}: {len(image_paths):5d}개")

    print(f"\n✅ 총 {len(file_paths):,}개 파일 경로 수집 완료!")

    return file_paths, labels, class_names

# 파일 경로만 수집 (메모리 절약!)
file_paths, labels, class_names = get_file_paths_and_labels(data_dir)

print(f"\n데이터 정보:")
print(f"  파일 수: {len(file_paths):,}개")
print(f"  클래스: {class_names}")
print(f"  메모리 사용: {len(file_paths) * 100 / 1024:.2f} KB (경로만)")

In [ ]:
# Train/Val Split (경로 기준)
from sklearn.model_selection import train_test_split

train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths, labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

print(f"Train: {len(train_paths):,}개")
print(f"Val  : {len(val_paths):,}개")

## 🎨 2. Data Augmentation for Transfer Learning

In [ ]:
# ImageNet Preprocessing 함수
def load_and_preprocess(path, label, augment=False):
    # 이미지 로드
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.image.resize(img, (224, 224))

    # RGB 변환
    img = tf.image.grayscale_to_rgb(img)

    # Data Augmentation (Training only)
    if augment:
        img = tf.image.random_flip_left_right(img)  # 좌우 반전은 조심!
        img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
        # 주의: 의료 이미지는 좌우 반전 금지가 원칙이지만,
        # 여기서는 실험적으로 적용 (실제로는 rotation만 권장)

    # ImageNet preprocess
    img = preprocess_input(img)

    return img, label

print("✅ Preprocessing 함수 정의 완료!")

In [ ]:
# tf.data.Dataset 생성
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Train Dataset (with augmentation)
train_dataset = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_dataset = train_dataset.map(
    lambda x, y: load_and_preprocess(x, y, augment=True),
    num_parallel_calls=AUTOTUNE
)
train_dataset = train_dataset.shuffle(1000)
train_dataset = train_dataset.batch(BATCH_SIZE)
train_dataset = train_dataset.prefetch(AUTOTUNE)

# Val Dataset (no augmentation)
val_dataset = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_dataset = val_dataset.map(
    lambda x, y: load_and_preprocess(x, y, augment=False),
    num_parallel_calls=AUTOTUNE
)
val_dataset = val_dataset.batch(BATCH_SIZE)
val_dataset = val_dataset.prefetch(AUTOTUNE)

print(f"✅ Dataset 생성 완료!")
print(f"   Train batches: {len(train_dataset)}")
print(f"   Val batches: {len(val_dataset)}")

## 🏗️ 3. ResNet50 Transfer Learning

### Phase 1: Feature Extraction

Pretrained ResNet50을 Feature Extractor로 사용  
→ Base model 동결, Classifier만 학습

In [ ]:
# ResNet50 Pretrained 로드
base_model = ResNet50(
    weights='imagenet',
    include_top=False,  # Classifier 제거
    input_shape=(224, 224, 3)
)

# Base model 동결
base_model.trainable = False

print(f"✅ ResNet50 로드 완료!")
print(f"   Layers: {len(base_model.layers)}")
print(f"   Trainable: {base_model.trainable}")

🔥 이 부분을 같이 작성해봅시다.

ResNet50 위에 올릴 **Custom Classifier**를 완성해 보세요.
GlobalAveragePooling → Dense(512) + BN + Dropout(0.5) → Dense(256) + BN + Dropout(0.3) → Dense(4, softmax)

In [ ]:
# Transfer Learning 모델 구축
inputs = keras.Input(shape=(224, 224, 3))

# Pretrained Feature Extractor (동결 상태)
x = base_model(inputs, training=False)

# Custom Classifier
x = layers.GlobalAveragePooling2D()(x)
x = # 🔥 직접 작성이 필요합니다. (Dense(512, relu) + BatchNorm + Dropout(0.5))
x = # 🔥 직접 작성이 필요합니다. (Dense(256, relu) + BatchNorm + Dropout(0.3))
outputs = # 🔥 직접 작성이 필요합니다. (Dense(4, softmax))

model = keras.Model(inputs, outputs, name='ResNet50_COVID')
model.summary()


In [ ]:
# 모델 컴파일
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ 모델 컴파일 완료!")

## 🏃 4. Phase 1: Feature Extraction 학습

In [ ]:
# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7
)

checkpoint = ModelCheckpoint(
    'best_model_phase1.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max'
)

print("✅ Callbacks 설정 완료!")

🔥 이 부분은 수정이 필요합니다.

**Phase 1 (Feature Extraction)** 실험의 **run_name**을 채워주세요.

In [ ]:
# Phase 1 학습
try:
    mlflow.end_run()
except:
    pass

with mlflow.start_run(run_name=''):  # 🔥 직접 작성이 필요합니다. (Phase 1 이름)
    mlflow.log_params({
        'model': 'ResNet50',
        'phase': 'Feature Extraction',
        'base_trainable': False,
        'optimizer': 'adam',
        'epochs': 15,
        'batch_size': BATCH_SIZE,
        'augmentation': True
    })

    print("🏃 Phase 1 학습 시작 (Feature Extraction)...\n")

    history_phase1 = model.fit(
        train_dataset,
        epochs=15,
        validation_data=val_dataset,
        callbacks=[early_stop, reduce_lr, checkpoint],
        verbose=1
    )

    # 최종 성능
    final_loss, final_acc = model.evaluate(val_dataset, verbose=0)

    mlflow.log_metrics({
        'phase1_val_loss': final_loss,
        'phase1_val_accuracy': final_acc
    })

    print(f"\n{'='*60}")
    print("  Phase 1 완료")
    print('='*60)
    print(f"  Val Accuracy: {final_acc:.4f} ({final_acc*100:.2f}%)")
    print('='*60)

## 🔧 5. Phase 2: Fine-tuning

### Fine-tuning 전략

Base model의 상위 레이어를 학습 가능하게 변경  
→ 낮은 Learning Rate로 미세 조정

🔥 이 부분을 같이 작성해봅시다.

Fine-tuning에서 **상위 몇 개의 레이어를 해제**할지 결정해 보세요. (권장: 30)

In [ ]:
# Base model 일부 해제
base_model.trainable = True

# 하위 레이어는 동결 유지 (상위 30개만 학습)
for layer in base_model.layers[:-# 🔥 직접 작성이 필요합니다. (예: 30)]:
    layer.trainable = False

print(f"✅ Fine-tuning 준비 완료!")
print(f"   Total layers: {len(base_model.layers)}")
print(f"   Trainable layers: {sum([1 for l in base_model.layers if l.trainable])}")

In [ ]:
# 재컴파일 (낮은 LR)
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),  # 매우 낮은 LR!
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ 낮은 Learning Rate로 재컴파일 완료!")

🔥 이 부분은 수정이 필요합니다.

**Phase 2 (Fine-tuning)** 실험의 **run_name**을 채워주세요.

In [ ]:
# Phase 2 학습
try:
    mlflow.end_run()
except:
    pass

with mlflow.start_run(run_name=''):  # 🔥 직접 작성이 필요합니다. (Phase 2 이름)
    mlflow.log_params({
        'model': 'ResNet50',
        'phase': 'Fine-tuning',
        'base_trainable': True,
        'trainable_layers': 30,
        'optimizer': 'adam',
        'learning_rate': 1e-5,
        'epochs': 10,
        'batch_size': BATCH_SIZE
    })

    print("🏃 Phase 2 학습 시작 (Fine-tuning)...\n")

    history_phase2 = model.fit(
        train_dataset,
        epochs=10,
        validation_data=val_dataset,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )

    # 최종 성능
    final_loss, final_acc = model.evaluate(val_dataset, verbose=0)

    mlflow.log_metrics({
        'phase2_val_loss': final_loss,
        'phase2_val_accuracy': final_acc
    })

    # 모델 저장
    model.save('resnet50_finetuned.h5')
    mlflow.keras.log_model(model, 'model')

    print(f"\n{'='*60}")
    print("  Phase 2 완료")
    print('='*60)
    print(f"  Val Accuracy: {final_acc:.4f} ({final_acc*100:.2f}%)")
    print('='*60)

## 📊 6. 성능 평가

In [ ]:
# 학습 곡선 (Phase 1 + Phase 2)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Phase 1 Loss
axes[0, 0].plot(history_phase1.history['loss'], label='Train')
axes[0, 0].plot(history_phase1.history['val_loss'], label='Val')
axes[0, 0].set_title('Phase 1 - Loss', fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Phase 1 Accuracy
axes[0, 1].plot(history_phase1.history['accuracy'], label='Train')
axes[0, 1].plot(history_phase1.history['val_accuracy'], label='Val')
axes[0, 1].set_title('Phase 1 - Accuracy', fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Phase 2 Loss
axes[1, 0].plot(history_phase2.history['loss'], label='Train')
axes[1, 0].plot(history_phase2.history['val_loss'], label='Val')
axes[1, 0].set_title('Phase 2 - Loss', fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Phase 2 Accuracy
axes[1, 1].plot(history_phase2.history['accuracy'], label='Train')
axes[1, 1].plot(history_phase2.history['val_accuracy'], label='Val')
axes[1, 1].set_title('Phase 2 - Accuracy', fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.suptitle('ResNet50 Transfer Learning — 학습 곡선',
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('training_curves_resnet50.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Validation 예측
print("🔮 Validation set 예측 중...")

y_true = []
y_pred = []
y_pred_proba = []

for images, labels in val_dataset:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    y_pred_proba.extend(preds)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_pred_proba = np.array(y_pred_proba)

print("✅ 예측 완료!")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.xlabel('예측', fontweight='bold', fontsize=12)
plt.ylabel('실제', fontweight='bold', fontsize=12)
plt.title('Confusion Matrix — ResNet50 Transfer Learning',
          fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrix_resnet50.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Classification Report
report = classification_report(y_true, y_pred,
                              target_names=class_names,
                              digits=4)

print("="*60)
print("  Classification Report — ResNet50")
print("="*60)
print(report)
print("="*60)

# Per-class Metrics
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None
)

print("\n클래스별 성능:")
print("-"*60)
for i, class_name in enumerate(class_names):
    print(f"{class_name:20s}: "
          f"Precision={precision[i]:.4f}, "
          f"Recall={recall[i]:.4f}, "
          f"F1={f1[i]:.4f}")
print("-"*60)

# COVID-19 Recall
covid_idx = class_names.index('COVID')
print(f"\n⚠️ COVID-19 Recall: {recall[covid_idx]:.4f} ({recall[covid_idx]*100:.2f}%)")
print(f"   → {int(recall[covid_idx]*support[covid_idx])}/{support[covid_idx]} COVID 환자 탐지")

## 📈 7. Baseline vs Transfer Learning 비교

In [ ]:
# 성능 비교
comparison = {
    'Model': ['Baseline CNN', 'ResNet50 Transfer'],
    'Accuracy': [0.8542, final_acc],
    'COVID Recall': [0.9571, recall[0]],
    'Viral Precision': [0.6059, precision[3]]
}

comp_df = pd.DataFrame(comparison)

print("="*60)
print("  Baseline vs Transfer Learning")
print("="*60)
print(comp_df.to_string(index=False))
print("="*60)

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['Accuracy', 'COVID Recall', 'Viral Precision']
baseline = [0.8542, 0.9571, 0.6059]
transfer = [final_acc, recall[0], precision[3]]

for i, (metric, ax) in enumerate(zip(metrics, axes)):
    bars = ax.bar(['Baseline', 'ResNet50'], [baseline[i], transfer[i]],
                  color=['steelblue', 'coral'], edgecolor='black')
    ax.set_ylabel(metric, fontweight='bold')
    ax.set_title(metric, fontweight='bold')
    ax.set_ylim(0.5, 1.0)
    ax.grid(axis='y', alpha=0.3)

    for bar, val in zip(bars, [baseline[i], transfer[i]]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.4f}', ha='center', fontweight='bold')

plt.suptitle('Baseline CNN vs ResNet50 Transfer Learning',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('baseline_vs_transfer.png', dpi=100, bbox_inches='tight')
plt.show()

## 🧠 8. 핵심 개념 정리

### 오늘 배운 것

**1. Transfer Learning**
- ImageNet Pretrained → COVID 데이터 전이
- Feature Extraction: Base 동결, Classifier만 학습
- Fine-tuning: 상위 레이어 미세 조정

**2. ResNet50 아키텍처**
- Skip Connection (Residual)
- 50개 레이어 (깊은 네트워크)
- ImageNet 1000-class 학습

**3. Data Augmentation**
- 회전, 이동, 확대/축소
- 의료 이미지 주의사항 (좌우 반전 금지)

**4. 2-Phase 학습**
- Phase 1: Feature Extraction (15 epochs)
- Phase 2: Fine-tuning (10 epochs, LR=1e-5)

**5. 성능 향상**
```
Baseline → ResNet50
Accuracy:        85.42% → 90%+
COVID Recall:    95.71% → 95%+ (유지)
Viral Precision: 60.59% → 75%+
```

---

### Day 4-3 예고

**Class Imbalance 처리**
- Class Weights 적용
- Focal Loss
- SMOTE Oversampling
- 목표: Viral Precision 80%+, 균형 잡힌 성능

축하합니다! Day 4-2 완료! 🎉

## ✅ Day 4-2 완료 체크리스트

- [ ] Transfer Learning 개념 이해
- [ ] ResNet50 Pretrained 로드
- [ ] Feature Extraction 학습
- [ ] Fine-tuning 학습
- [ ] Data Augmentation 적용
- [ ] Learning Rate Scheduling
- [ ] Phase 1 + Phase 2 학습 곡선
- [ ] Confusion Matrix 분석
- [ ] Classification Report 확인
- [ ] Baseline과 성능 비교
- [ ] Val Accuracy 90%+ 달성
- [ ] MLflow 기록 완료

## 🎯 다음 단계 (Day 4-3)

**Day 4-3: Class Imbalance 처리**

**내용:**
- Class Weights 계산 및 적용
- Focal Loss 구현
- SMOTE Oversampling
- Undersampling 전략
- 균형 잡힌 성능 달성

**목표:**
- Viral Pneumonia Precision: 80%+
- 모든 클래스 F1 > 0.85
- Balanced Accuracy 향상

수고하셨습니다! 🚀